# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, based on its [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print dataset title and description
print(f"{getattr(metadata, 'name', 'Dataset')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.record_sets)
print("Available record sets and their @ids:")
for rs in record_sets:
    print(f"  - {rs['@id']} (name: {rs.get('name', '')})")

In [ ]:
# For each record set, list its fields with their @id and data types
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name', '')})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for f in fields:
        if isinstance(f, str):
            # If only @id was provided
            f_detail = dataset.get_by_id(f)
        else:
            f_detail = f
        if f_detail is None:
            continue
        fid = f_detail.get('@id', '<unknown>')
        name = f_detail.get('name', '')
        dtype = f_detail.get('dataType', '')
        print(f"    - {fid} (name: {name}, type: {dtype})")

## 3. Data Extraction
Load data from a chosen record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For demonstration, we'll load the first tabular record set found
tabular_record_sets = [rs for rs in record_sets if rs.get('@type') == 'cr:RecordSet']
if not tabular_record_sets:
    print("No record sets of type 'cr:RecordSet' found.")
else:
    # Use the first available as an example
    record_set_id = tabular_record_sets[0]['@id']
    print(f"Extracting data from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print('Available columns:')
    print(list(df.columns))
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's explore numeric fields, apply filtering, normalization, and grouping. We use the field `@id`s for all references.

In [ ]:
# Choose a numeric field from the data, using its @id. (Adjust as appropriate.)
numeric_fields = []
fields_detail = tabular_record_sets[0].get('field', [])
if isinstance(fields_detail, dict):
    fields_detail = [fields_detail]
for f in fields_detail:
    if isinstance(f, str):
        f = dataset.get_by_id(f)
    if f is not None and f.get('dataType') in ['schema:Number', 'schema:Integer', 'schema:Float']:
        numeric_fields.append(f.get('@id'))

if not numeric_fields:
    print('No numeric fields to analyze.')
else:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
    if numeric_field_id in df:
        # Attempt convert to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        if df[numeric_field_id].std(skipna=True) > 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        else:
            print(f"{numeric_field_id} could not be normalized (std=0 or missing values).")

        # Grouping by a categorical field (first non-numeric)
        other_fields = [f.get('@id') for f in fields_detail if f.get('dataType') not in ['schema:Number', 'schema:Integer', 'schema:Float']]
        group_field = other_fields[0] if other_fields else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print('No suitable group field found for grouping.')
    else:
        print(f"Chosen field ({numeric_field_id}) not in DataFrame columns.")

## 5. Visualization
Visualize the distribution of the selected numeric field. We'll use matplotlib for a histogram if possible.

In [ ]:
import matplotlib.pyplot as plt

if not numeric_fields:
    print('No numeric fields to plot.')
else:
    field_to_plot = numeric_field_id
    if field_to_plot in df.columns and df[field_to_plot].dtype.kind in 'fi':
        plt.figure(figsize=(7,4))
        df[field_to_plot].hist(bins=20)
        plt.xlabel(field_to_plot)
        plt.ylabel('Count')
        plt.title(f'Distribution of {field_to_plot}')
        plt.show()
    else:
        print(f'Field {field_to_plot} not numeric or not present for plotting.')

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to discover metadata, enumerate all record sets and fields by `@id`, extract structured data, and perform basic exploratory data analyses with proper `@id` referencing. See the [FAIR^2 dataset page](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for more documentation, licensing, and scientific context.

_You can adapt these steps to suit any Croissant-compliant dataset by supplying the new `@id` values for record sets and fields as needed._